In [44]:
import DataPrep
import keras
import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np

In [71]:
name="dingus2"


tfrecordpath = "../Data/tfrecords/"

windowsize = 150
lookahead = 5
batch_size = 1

coins = ["DOTUSD_PERP"]
datasets = DataPrep.getAllSliders(coins, windowsize, lookahead, batch_size, tfrecordpath)
valdataset = datasets.pop(-1)


folder = "models/" + name + "/"
# Load model
model = keras.models.load_model(folder + "model.keras")

checkpoint = tf.train.Checkpoint(model=model)
manager = tf.train.CheckpointManager(checkpoint=checkpoint, directory=folder, max_to_keep=3)
manager.restore_or_initialize()


/root/miniconda3/envs/levbot/lib/python3.11/site-packages/keras/src/saving/saving_lib.py:757: UserWarning: Skipping variable loading for optimizer 'adam', because it has 50 variables whereas the saved optimizer has 2 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


'models/dingus2/ckpt-4'

In [86]:
example = valdataset.__iter__().__next__()
ready4model = DataPrep.createLabelsBatch(example[0], example[1])
prediction = model.predict(ready4model[0])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step


In [87]:
print(example[1][0].shape)
baseprice = example[1][0,0,0,0]
print(f"Base Price: {baseprice:.4}")
realhighs = example[1][0,0,1,:]
reallows = example[1][0,0,2,:]
maxes = np.max(reallows), np.max(realhighs)
predictions = ((prediction[0]/ 10) + 1)*baseprice
print(maxes)
print(predictions)
difference = predictions[0] - realhighs[0], predictions[1] - realhighs[1]
print(f"differencelow: {difference[0]/baseprice * 100}")
print(f"differencehigh: {difference[1]/baseprice * 100}")



(6, 5, 6)
Base Price: 9.373
(9.383, 9.395)
tf.Tensor([9.35492  9.391888], shape=(2,), dtype=float32)
differencelow: -0.23556455969810486
differencehigh: 0.08415491133928299
